# Week 2 Coding Assignment: NumPy Practice

**Data Science: Foundations to Applications | Kyungdong University, Global Campus**

**Student:** Daniel Garang Nyok Garang (2317394)

**Outcome being assessed:** Write vectorized Python/NumPy code instead of manual loops.

## Instructions
- Complete every function below by replacing `# TODO` with your code.
- **Do not use a `for` or `while` loop anywhere in Parts A–C** — every problem can be solved with vectorized NumPy operations.
- Run the test cell below each function to check your answer.
- When finished: save the notebook, then `git add`, `git commit`, and `git push` to your `data-science-labs-2026` repository, per the Week 1 lab.

**Parts:**
- Part A — Array Manipulation (Problems 1–4)
- Part B — Matrix Operations (Problems 5–7)
- Part C — Simple Simulations (Problems 8–10)
- Stretch Challenge (optional)

> ⚠️ **Run this notebook in Jupyter Notebook (via Anaconda) — not Google Colab.**

In [1]:
import numpy as np
np.random.seed(42)  # keep this seed so your results are reproducible

---
## Part A — Array Manipulation

### Problem 1 — Temperature Converter
You work for a weather-data startup expanding into the US market. Convert an array of Celsius temperatures to Fahrenheit using the formula `F = C * 9/5 + 32`.

**Hint:** This is pure element-wise arithmetic — no loop needed.

In [2]:
def celsius_to_fahrenheit(celsius_array):
    """
    celsius_array: 1D NumPy array of temperatures in Celsius
    returns: 1D NumPy array of temperatures in Fahrenheit
    """
    # Arithmetic on a NumPy array is applied to every element at once (broadcasting)
    return celsius_array * 9 / 5 + 32

# Test
test_c = np.array([0, 20, 37, 100])
print(celsius_to_fahrenheit(test_c))  # expected: [ 32.   68.   98.6 212. ]

[ 32.   68.   98.6 212. ]


### Problem 2 — Filter High-Value Transactions
A fintech app logs transaction amounts (in KRW). Return only the transactions greater than or equal to 50,000 KRW.

**Hint:** Boolean masking.

In [3]:
def filter_high_value(transactions, threshold=50000):
    """
    transactions: 1D NumPy array of transaction amounts
    threshold: minimum amount to keep
    returns: 1D NumPy array containing only transactions >= threshold
    """
    mask = transactions >= threshold   # array of True/False, one per transaction
    return transactions[mask]          # keep only the positions where mask is True

# Test
test_tx = np.array([12000, 55000, 49999, 100000, 50000])
print(filter_high_value(test_tx))  # expected: [ 55000 100000  50000]

[ 55000 100000  50000]


### Problem 3 — Reshape Sensor Data
An IoT sensor logs one temperature reading every hour for 7 days (168 readings total), stored as a flat 1D array. Reshape it into a `(7, 24)` array so each row represents one day.

**Hint:** `.reshape()`

In [4]:
def reshape_to_days(readings):
    """
    readings: 1D NumPy array of length 168
    returns: 2D NumPy array of shape (7, 24)
    """
    # -1 lets NumPy work out the number of days (168 / 24 = 7)
    return readings.reshape(-1, 24)

# Test
test_readings = np.arange(168)
result = reshape_to_days(test_readings)
print("shape:", result.shape)  # expected: (7, 24)
print("day 1:", result[0])     # expected: [0 1 2 ... 23]

shape: (7, 24)
day 1: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


### Problem 4 — Daily Min/Max Temperature
Using the reshaped `(7, 24)` sensor array from Problem 3, find the minimum and maximum temperature for each day in a single vectorized call each (no loop over days).

**Hint:** Use the `axis` parameter.

In [5]:
def daily_min_max(daily_readings):
    """
    daily_readings: 2D NumPy array of shape (n_days, 24)
    returns: tuple of two 1D arrays (daily_min, daily_max), each of length n_days
    """
    # axis=1 collapses the 24 hourly columns, leaving one value per row (day)
    daily_min = daily_readings.min(axis=1)
    daily_max = daily_readings.max(axis=1)
    return daily_min, daily_max

# Test
test_data = reshape_to_days(np.arange(168))
mins, maxs = daily_min_max(test_data)
print("daily mins:", mins)  # expected: [  0  24  48  72  96 120 144]
print("daily maxs:", maxs)  # expected: [ 23  47  71  95 119 143 167]

daily mins: [  0  24  48  72  96 120 144]
daily maxs: [ 23  47  71  95 119 143 167]


---
## Part B — Matrix Operations

### Problem 5 — Total Revenue by Product
A store tracks `units_sold` (shape `(n_days, n_products)`) and a `unit_price` vector (shape `(n_products,)`). Compute the total revenue per product across all days using matrix operations (no loop over days or products).

**Hint:** Think about `.sum(axis=...)` combined with element-wise or matrix multiplication.

In [6]:
def total_revenue_by_product(units_sold, unit_price):
    """
    units_sold: 2D array, shape (n_days, n_products)
    unit_price: 1D array, shape (n_products,)
    returns: 1D array, shape (n_products,) — total revenue per product
    """
    # Broadcasting: unit_price (n_products,) is stretched across every day (row),
    # giving daily revenue per product. Summing over axis=0 adds up the days.
    return (units_sold * unit_price).sum(axis=0)
    # Equivalent: units_sold.sum(axis=0) * unit_price

# Test
test_units = np.array([[10, 5], [8, 6], [12, 4]])  # 3 days, 2 products
test_price = np.array([1000, 2000])
print(total_revenue_by_product(test_units, test_price))  # expected: [30000 30000]

[30000 30000]


### Problem 6 — Grade Book Weighted Average
A grade book stores raw scores as a matrix `scores` of shape `(n_students, n_assignments)`, and a `weights` vector of shape `(n_assignments,)` that sums to 1.0. Compute each student's final weighted score using matrix multiplication.

**Hint:** `@`

In [7]:
def weighted_final_scores(scores, weights):
    """
    scores: 2D array, shape (n_students, n_assignments)
    weights: 1D array, shape (n_assignments,), sums to 1.0
    returns: 1D array, shape (n_students,) — each student's weighted final score
    """
    # (n_students, n_assignments) @ (n_assignments,) -> (n_students,)
    # Each result is the dot product: sum(score_i * weight_i) for one student.
    return scores @ weights

# Test
test_scores = np.array([[90, 80, 70], [60, 70, 80], [100, 90, 95]])
test_weights = np.array([0.5, 0.3, 0.2])
print(weighted_final_scores(test_scores, test_weights))  # expected: [83. 67. 96.]

[83. 67. 96.]


### Problem 7 — Solve for Ingredient Cost
A bakery sells two products. Product A uses 2 units of flour and 1 unit of sugar; Product B uses 1 unit of flour and 3 units of sugar. Product A costs 3,000 KRW and Product B costs 4,000 KRW in ingredients. Set up and solve the linear system to find the cost per unit of flour and sugar.

**Hint:** `np.linalg.solve`. Set up the coefficient matrix and results vector yourself.

**Setting up the system** (f = cost of one unit of flour, s = cost of one unit of sugar):

$$2f + 1s = 3000 \quad\text{(Product A)}$$
$$1f + 3s = 4000 \quad\text{(Product B)}$$

In matrix form $A\mathbf{x} = \mathbf{b}$ with $A = \begin{bmatrix}2 & 1\\ 1 & 3\end{bmatrix}$, $\mathbf{x} = \begin{bmatrix}f\\ s\end{bmatrix}$, $\mathbf{b} = \begin{bmatrix}3000\\ 4000\end{bmatrix}$.

In [8]:
def solve_ingredient_costs():
    """
    returns: 1D array [cost_per_flour_unit, cost_per_sugar_unit]
    """
    A = np.array([[2, 1],    # Product A: 2 flour + 1 sugar
                  [1, 3]])   # Product B: 1 flour + 3 sugar
    b = np.array([3000, 4000])
    return np.linalg.solve(A, b)

print(solve_ingredient_costs())  # expected: [1000. 1000.]
# Check: 2*1000 + 1*1000 = 3000 and 1*1000 + 3*1000 = 4000

[1000. 1000.]


---
## Part C — Simple Simulations

### Problem 8 — Coin Flip Simulation
Simulate flipping a fair coin 50,000 times (`0` = tails, `1` = heads) and return the proportion of heads.

**Hint:** `np.random.randint(0, 2, size=...)` and `.mean()`.

In [9]:
def simulate_coin_flips(n_flips=50000, seed=42):
    """
    returns: float, proportion of flips that were heads (close to 0.5)
    """
    np.random.seed(seed)
    flips = np.random.randint(0, 2, size=n_flips)   # 0 or 1 (upper bound 2 is excluded)
    # The mean of 0/1 values is exactly the proportion of 1s (heads)
    return float(flips.mean())

print(simulate_coin_flips())  # expected: a value close to 0.5

0.49908


### Problem 9 — Customer Wait Time Simulation
A call center's wait times are normally distributed with mean 5 minutes and standard deviation 1.5 minutes. Simulate 10,000 customer wait times and return the percentage of customers who waited longer than 7 minutes.

**Hint:** `np.random.normal`, then a boolean mask + `.mean() * 100`.

In [10]:
def pct_waiting_over_7min(n_customers=10000, seed=42):
    """
    returns: float, percentage (0-100) of customers who waited > 7 minutes
    """
    np.random.seed(seed)
    wait_times = np.random.normal(loc=5, scale=1.5, size=n_customers)
    # (wait_times > 7) is True/False; its mean is the fraction that is True
    return float((wait_times > 7).mean() * 100)

print(pct_waiting_over_7min())  # expected: roughly 9-10 (percent)
# Theory: z = (7 - 5) / 1.5 = 1.33, and P(Z > 1.33) ≈ 9.1%

9.229999999999999


### Problem 10 — Simple Portfolio Growth Simulation
Simulate 252 trading days (1 year) of daily returns for a stock, where daily returns are normally distributed with mean 0.0005 and standard deviation 0.015. Starting from an investment of 1,000,000 KRW, return the final portfolio value after compounding all 252 days.

**Hint:** `np.cumprod`, same pattern shown in the demo notebook's random walk example.

In [11]:
def simulate_portfolio_growth(start_value=1_000_000, n_days=252, seed=42):
    """
    returns: float, final portfolio value after n_days of compounded daily returns
    """
    np.random.seed(seed)
    daily_returns = np.random.normal(loc=0.0005, scale=0.015, size=n_days)
    # Compounding: value_t = start * (1+r1) * (1+r2) * ... * (1+rt)
    growth_path = np.cumprod(1 + daily_returns)      # growth factor after each day
    portfolio_values = start_value * growth_path      # value after each day
    return float(portfolio_values[-1])                # value after the last day

print(simulate_portfolio_growth())  # your value should be reproducible with the given seed

1089172.0041520006


---
## Stretch Challenge (Optional)
Simulate 1,000 independent random walks of a stock price (252 days each, same distribution as Problem 10), producing a `(1000, 252)` array of prices — all without any loop. Then report:
1. The mean final price across all 1,000 simulations
2. The percentage of simulations that ended below the starting value

This is the core idea behind **Monte Carlo simulation**.

In [12]:
def monte_carlo_portfolio(n_simulations=1000, start_value=1_000_000, n_days=252, seed=42):
    """
    returns: tuple (mean_final_price, pct_below_start)
    """
    np.random.seed(seed)
    # One row per simulation, one column per trading day
    daily_returns = np.random.normal(0.0005, 0.015, size=(n_simulations, n_days))
    # axis=1 -> compound along each row (each simulation) separately
    prices = start_value * np.cumprod(1 + daily_returns, axis=1)   # shape (1000, 252)

    final_prices = prices[:, -1]                       # last day of every simulation
    mean_final_price = float(final_prices.mean())
    pct_below_start = float((final_prices < start_value).mean() * 100)
    return mean_final_price, pct_below_start

print(monte_carlo_portfolio())

(1133520.334959366, 33.4)
